# Filtering and forecasting with uncertainty

Two screamer operators that carry uncertainty: `KalmanFilter` recovers a smooth latent signal from noisy observations, and `BayesianRegression` predicts the next value from a feature while reporting a calibrated interval around that prediction.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from screamer import KalmanFilter, BayesianRegression

rng = np.random.default_rng(0)

def show(ax, title, xlabel="sample"):
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.legend(fontsize=8)
    plt.tight_layout()

## Recovering a latent signal with the Kalman filter

The Kalman filter assumes the underlying level follows a random walk and that each observation adds independent noise on top of it. At every step it forms a weighted blend of the prior prediction and the new measurement, where the weights depend on how much it trusts the model versus the sensor. The filtered output tracks the true level without ever seeing it directly.

In [ ]:
N = 400
t = np.linspace(0, 4 * np.pi, N)
true_signal = np.sin(t) + 0.4 * np.sin(2.3 * t)
sigma_obs = 1.0
obs = true_signal + sigma_obs * rng.standard_normal(N)

filtered = KalmanFilter(process_var=0.01, observation_var=1.0)(obs)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(obs, lw=0.6, color="0.78", label="noisy observations")
ax.plot(true_signal, lw=1.2, color="0.35", ls="--", label="true signal")
ax.plot(filtered, lw=1.8, color="steelblue",
        label="KalmanFilter(process_var=0.01, observation_var=1.0)")
show(ax, "Kalman filter: recovering the latent signal")

## The process/observation noise trade-off

`process_var` controls how much the filter expects the latent level to drift between steps; `observation_var` controls how noisy the measurements are. A high `observation_var` relative to `process_var` makes the filter trust its own prediction more than each new reading, giving a smoother but laggier output. A high `process_var` does the opposite: it follows the data closely and responds fast, at the cost of passing more measurement noise through.

In [ ]:
responsive = KalmanFilter(process_var=1.0, observation_var=1.0)(obs)
smooth     = KalmanFilter(process_var=0.001, observation_var=1.0)(obs)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(obs, lw=0.6, color="0.78", label="noisy observations")
ax.plot(true_signal, lw=1.2, color="0.35", ls="--", label="true signal")
ax.plot(responsive, lw=1.6, color="tomato",
        label="responsive (process_var=1.0, observation_var=1.0)")
ax.plot(smooth, lw=1.6, color="steelblue",
        label="smooth (process_var=0.001, observation_var=1.0)")
show(ax, "Kalman filter: responsive vs smooth settings")

## Online forecasting with BayesianRegression

`BayesianRegression` fits a linear model `y = slope * x + intercept + noise` online using a conjugate Normal-Inverse-Gamma posterior. It takes two inputs `(y, x)` and returns four outputs per step: the posterior slope, intercept, a causal one-step-ahead predictive mean, and a predictive standard deviation. The prediction is formed before the posterior is updated with the current observation, so no future data leaks in. Because the prior is weak and proper, `pred_std` is finite from the very first step.

In [ ]:
n = 500
true_slope = 1.5
true_intercept = 0.3
x = rng.standard_normal(n)
y = true_slope * x + true_intercept + 0.6 * rng.standard_normal(n)

out = BayesianRegression(span=80)(y, x)
slope_est   = out[:, 0]
intercept_est = out[:, 1]
pred_mean   = out[:, 2]
pred_std    = out[:, 3]

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.scatter(range(n), y, s=4, color="0.70", label="y (observed)", zorder=1)
ax.plot(pred_mean, lw=1.8, color="darkorange", label="pred_mean", zorder=3)
ax.fill_between(range(n),
                pred_mean - 2 * pred_std,
                pred_mean + 2 * pred_std,
                color="darkorange", alpha=0.20,
                label="pred_mean +/- 2 * pred_std")
show(ax, "BayesianRegression: causal prediction with uncertainty band (span=80)")

Early in the stream the posterior has seen only a few points, so the predictive band is wide. As evidence accumulates the band narrows and the recovered slope converges toward the true value. The orange band covers roughly 95% of observations once it has settled, which is what a calibrated two-standard-deviation interval should do.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.0))
ax.axhline(true_slope, color="0.35", ls="--", lw=1.2, label=f"true slope = {true_slope}")
ax.plot(slope_est, lw=1.8, color="darkorange", label="estimated slope")
ax.set_ylim(true_slope - 1.5, true_slope + 1.5)
show(ax, "Posterior slope converging to the true value")